<a href="https://colab.research.google.com/github/BosunL/SEA-VQA/blob/main/Phi_3_Vision_SEA_CVQA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Visual Question Answering — Phi-3 Vision 128K

Evaluating `microsoft/Phi-3-vision-128k-instruct` on a custom Kenyan cultural dataset: 50 questions × 3 languages (English, Ateso, Swahili) across 14 images.

### 1. Install Necessary Libraries

In [ ]:
!pip install -q transformers==4.40.2 accelerate pillow matplotlib requests einops
#!pip install -q transformers accelerate pillow matplotlib requests einops


### 2. Import Libraries

In [ ]:
# STEP 1: Import Libraries
import random, io, requests, torch
import matplotlib.pyplot as plt
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM
from huggingface_hub import login

print("Libraries imported successfully.")


### 3. Authenticate with Hugging Face

Accept the Phi-3 Vision license at https://huggingface.co/microsoft/Phi-3-vision-128k-instruct then run this cell.

In [ ]:
# ============================================================
# STEP 2: Authenticate with Hugging Face
# ============================================================
# Accept the Phi-3 Vision license at:
# https://huggingface.co/microsoft/Phi-3-vision-128k-instruct
# Then paste your HF token below.

login(token="INSERT TOKEN")


### 4. Load Model and Processor

In [ ]:
# ============================================================
# STEP 3: Load Model and Processor
# ============================================================

print("Loading Phi-3 Vision 128K model and processor...")

MODEL_ID = "microsoft/Phi-3-vision-128k-instruct"

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    _attn_implementation="eager",   # use "flash_attention_2" only if flash_attn is installed
)

print(f"Model loaded on: {model.device}\n")


### 5. Load Custom Dataset

All 50 questions in English, Ateso, and Swahili. Images are fetched from Google Drive by file ID and cached so each image downloads only once.

In [ ]:
# ============================================================
# STEP 4: Load Custom Kenyan Culture Dataset (English + Ateso + Swahili)
# ============================================================
# Images are loaded from Google Drive using the file ID.
# A cache dict avoids re-downloading the same image multiple times.

CUSTOM_DATA = [
    {
        "ID": "custom_001",
        "Category": "Image 1 Questions",
        "file_id": "14pBCQkb9v2VCsBEsE5NWlpMHrWx_FoN-",
        "eng_question": "Which of the baskets are pictured, what are they called in this country, and what is their purpose?",
        "correct_en": "Uteo. Crafted from reeds or palm leaves. It is traditionally found in households and was used to separate the chaff from grain.Its commonly used to hold cereals, in market places, and in homes.",
        "wrong_en_o1": "Mesob baskets. Individual strands of grass or palm are dyed with vibrant colors and utilized for communal serving holding traditional meals",
        "wrong_en_o2": "Bolga baskets. Made from thick, dried elephant grass. They are also very durable because of the tight weaving and are used as market or beach bags, home decor, and for storage and organization.",
        "wrong_en_o3": "Emotokaa. Kanueraitere nakapoloni itomate adakia inyamata luipwaka katoni luitijimete inyamene ayari namakegwelete",
        "native_question": "Ibukito alu ajsii aputo, enayrit kesi nyena kakwap kalo? Inyena nesii asoma kesii?",
        "correct_native": "Euderu. Ejeo kaladoi kiton amusogon edumunun  kotogoi lukalong itosomao akipiet inyamen Itosomao adakite eariot kosokoni katoni kore",
        "wrong_native_o1": "Ibukit luka mesob. Anyart apee akou kanyaiti kape isutite, iranginino luchaete na itomayo akiria kadakite inyamene katikere",
        "wrong_native_o2": "Ibukito luka bolga. Ejeo kesi alumni kosere kaleoburo cuti. Ichasit kesi aso iboete cuti aoaki neuja.  Asoma kesi kogwelete kanamikwangere akilimia itogoi, akigwa iboro,  katoni atongitongoi",
        "wrong_native_o3": "Eguniati ngoli titosmomao akitowonio ichoko luepelel ingarakini akitene ichoko lukakiraa kotetenyi alumni iraito lukejoka",
        "swa_question": "Ni vikapu gani kati ya hivyo vilivyoonyeshwa, vinaitwaje katika nchi hii, na kusudi lao ni lipi?",
        "correct_swa": "Uteo. Imetengenezwa kwa matete au majani ya mitende. Kijadi hupatikana majumbani na ilitumika kutenganisha makapi na nafaka. Kwa kawaida hutumika kuhifadhi nafaka, sokoni, na majumbani.",
        "wrong_swa_o1": "Vikapu vya Mesob. Vipande vya nyasi au mitende hupakwa rangi angavu na hutumika kwa huduma ya pamoja ya kuandalia milo ya kitamaduni.",
        "wrong_swa_o2": "Vikapu vya Bolga. Vimetengenezwa kwa nyasi nene na kavu. Pia ni vya kudumu sana kwa sababu ya kusuka kwa ukali na hutumika kama mifuko ya sokoni au ufukweni, mapambo ya nyumbani, na kwa ajili ya kuhifadhi na kupanga.",
        "wrong_swa_o3": "Emotokaa. Kanueraitere nakapoloni itomate adakia inyamata luipwaka katoni luitijimete inyamene ayari namakegwelete.",
    },
    {
        "ID": "custom_002",
        "Category": "Image 1 Questions",
        "file_id": "14pBCQkb9v2VCsBEsE5NWlpMHrWx_FoN-",
        "eng_question": "What are the yellow jerry cans used for?",
        "correct_en": "Storage of cooking oils, water, and cereals",
        "wrong_en_o1": "Easy transportation of baskets",
        "wrong_en_o2": "They are used to store dough",
        "wrong_en_o3": "Prolong the shelf life of fresh fish and raw meat",
        "native_question": "Inyena asoma kejerikana nukobegete",
        "correct_native": "Akigwa akinyet kedia, akipii katoni inyamet luewokito",
        "wrong_native_o1": "Etapana adakari ibukito",
        "wrong_native_o2": "Itosomao akigwa akiria nu esudite",
        "wrong_native_o3": "Akituuja akijara kikole luepalala ka kiringi",
        "swa_question": "Makopo ya njano ya jerry hutumika kwa ajili ya nini?",
        "correct_swa": "Uhifadhi wa mafuta ya kupikia, maji, na nafaka",
        "wrong_swa_o1": "Usafirishaji rahisi wa vikapu",
        "wrong_swa_o2": "Zinatumika kuhifadhi unga",
        "wrong_swa_o3": "Ongeza muda wa matumizi ya samaki mbichi na nyama mbichi",
    },
    {
        "ID": "custom_003",
        "Category": "Image 1 Questions",
        "file_id": "14pBCQkb9v2VCsBEsE5NWlpMHrWx_FoN-",
        "eng_question": "What form of transportation is common for small-scale market vendors?",
        "correct_en": "Bicycle. It is highly affordable since it doesn't depend on fuel. It easily carries smaller harvests to the market, allowing vendors to sell their goods and travel light on the way back.",
        "wrong_en_o1": "Tuktuk. It is affordable and allows for easy transport of many items.",
        "wrong_en_o2": "Car. Since it’s the biggest, market vendors often transport mass amounts of food and spices from recent harvest to sale at the market.",
        "wrong_en_o3": "Boda-boda. The two-wheeled motorcycle or bicycle taxi allows for quickest transport to waiting customers and are found in many rural and urban areas.",
        "native_question": "Inyena itosomaete itugna adakaia iboro kogwelete kaluchisi.",
        "correct_native": "Akarigo. Atapana cuti akitosom kanukamum akinyet . Edaki idwenyana idisi na elosete itunga lukegwelete katapan abongori ore.",
        "wrong_native_o1": "Atuktuk. Itapana  etachiti kene na  etapana adakia iboro luepu.",
        "wrong_native_o2": "Emotokaa. Kanueraitere nakapoloni itomate adakia inyamata luipwaka katoni luitijimete inyamene ayari namakegwelete",
        "wrong_native_o3": "Ebodaboda. Eraiti achakadu ikokorori kiyare kere  akarigo na elosi katipe noi. Itosomate kitunga kalidarito alosite. Edumuni cuti kochalo kaoni otauni.",
        "swa_question": "Ni aina gani ya usafiri ambayo ni ya kawaida kwa wachuuzi wadogo wa soko?",
        "correct_swa": "Baiskeli. Ni nafuu sana kwa sababu haitegemei mafuta. Inasafirisha mavuno madogo sokoni kwa urahisi, na hivyo kuruhusu wachuuzi kuuza bidhaa zao na kusafiri kwa urahisi wanaporudi.",
        "wrong_swa_o1": "Tuktuk. Ni nafuu na inaruhusu usafirishaji rahisi wa vitu vingi.",
        "wrong_swa_o2": "Gari. Kwa kuwa ndilo kubwa zaidi, wachuuzi wa soko mara nyingi husafirisha kiasi kikubwa cha chakula na viungo kutoka kwa mavuno ya hivi karibuni hadi mauzo sokoni.",
        "wrong_swa_o3": "Boda-boda. Pikipiki au teksi ya baiskeli yenye magurudumu mawili inaruhusu usafiri wa haraka zaidi kwa wateja wanaosubiri na inapatikana katika maeneo mengi ya vijijini na mijini.",
    },
    {
        "ID": "custom_004",
        "Category": "Image 1 Questions",
        "file_id": "14pBCQkb9v2VCsBEsE5NWlpMHrWx_FoN-",
        "eng_question": "What is the significance of the sack?",
        "correct_en": "The sack is often used to store cassava and maize. The bags can be made from plastic and a tightly woven outer layer.",
        "wrong_en_o1": "The sack is frequently used to hold rice, which is cooked as a daily meal and mass-produced.",
        "wrong_en_o2": "The sack usually hold dried beans and better preserves nutrients and quality.",
        "wrong_en_o3": "The sack is used for drying wet seeds. It helps prepare the seeds to be planted, leading to better quality crops.",
        "native_question": "Nyena adumunet kegunieti?",
        "correct_native": "Eguniete etosomao akigwa emogo kekurididi. Itenio kaplastic katoni namaejaite kokinga baba",
        "wrong_native_o1": "Eguniati Etosomao akidaka emuchere loipoyo nginiparan kapolou",
        "wrong_native_o2": "Eguniati kachuti edakiti amaragwe nakaoni kakigwa tenan",
        "wrong_native_o3": "Eguniati ngoli titosmomao akitowonio ichoko luepelel ingarakini akitene ichoko lukakiraa kotetenyi alumni iraito lukejoka",
        "swa_question": "Umuhimu wa gunia ni upi?",
        "correct_swa": "Gunia mara nyingi hutumika kuhifadhi mihogo na mahindi. Mifuko inaweza kutengenezwa kwa plastiki na safu ya nje iliyosokotwa vizuri.",
        "wrong_swa_o1": "Gunia hilo hutumika mara nyingi kuhifadhi wali, ambao hupikwa kama mlo wa kila siku na hutengenezwa kwa wingi.",
        "wrong_swa_o2": "Kwa kawaida gunia huhifadhi maharagwe makavu na huhifadhi virutubisho na ubora zaidi.",
        "wrong_swa_o3": "Gunia hutumika kukaushia mbegu zenye unyevu. Husaidia kuandaa mbegu za kupanda, na hivyo kusababisha mazao bora zaidi.",
    },
    {
        "ID": "custom_005",
        "Category": "Image 2 Questions",
        "file_id": "1x8AKBikOaDExx_LhB-vvuvLQZfuYBpeM",
        "eng_question": "The blue container in the top right corner holds many dried, chipped pieces of a particular food. What food is made from this material, and what process was used to get to this point?",
        "correct_en": "Porridge. It is laid on the floor in thin layers on top of a cloth, left to be sundried, and then broken up into smaller pieces.",
        "wrong_en_o1": "Bread. They are battered in flour, dipped in frying oil, and then let rest until they are cooled.",
        "wrong_en_o2": "Chips. They are sliced into thin pieces, baked in an oven at a high heat until all of their moisture is gone, then dipped in water to cool off.",
        "wrong_en_o3": "Fried rice. The uncooked grains are laid in a row, flattened with a roller, and boiled.",
        "native_question": "Abesin nejii kowai kamaa kateten erumit inyamat. Inyamt ngulu itente kinyena, eponee ali litenitere?",
        "correct_native": "Akimaa. Ipikino pwap  aadis adis kuju kakaratasit, kakinyekin  kowokoto kakolong, kosodi kobilibilite, idis dis.",
        "wrong_native_o1": "Emugati. Itenio kakiria  kipikinete akinyet numwaka, kere konyee kinyekinete kobu.",
        "wrong_native_o2": "Ichips. Etubio idisdis kakipikini toma  Okiala loka kim loko kuju chut,  paka nam edaunore akipee kijokis,  kakipikin akipii kotauchi.",
        "wrong_native_o3": "Emuchere luwowate kakinyet. Emuchere nuh mema kipoote  epyakion kwapu, kakichikakin kakitoi, kakipore",
        "swa_question": "Chombo cha bluu kilicho kwenye kona ya juu kulia kina vipande vingi vya chakula fulani vilivyokauka na kukatwakatwa. Ni chakula gani kinachotengenezwa kutokana na nyenzo hii, na ni mchakato gani uliotumika kufikia hatua hii?",
        "correct_swa": "Uji. Huwekwa sakafuni katika tabaka nyembamba juu ya kitambaa, huachwa kikauke juani, kisha huvunjwa vipande vidogo.",
        "wrong_swa_o1": "Mkate. Huchanganywa katika unga, huchovya katika mafuta ya kukaangia, kisha huachwa zipumzike hadi zipoe.",
        "wrong_swa_o2": "Chipsi. Hukatwa vipande vipande nyembamba, huokwa katika oveni kwa moto mkali hadi unyevunyevu wote utoke, kisha huchovya kwenye maji ili kupoa.",
        "wrong_swa_o3": "Wali wa kukaanga. Nafaka ambazo hazijapikwa huwekwa mfululizo, husagwa kwa kutumia rola, na kuchemshwa.",
    },
    {
        "ID": "custom_006",
        "Category": "Image 2 Questions",
        "file_id": "1x8AKBikOaDExx_LhB-vvuvLQZfuYBpeM",
        "eng_question": "The preparations and consumption of many of the foods found in markets incorporate spices and seasonings such as cumin, turmeric, coriander, and ginger. This profile of foods reflects which historical trading relationship?",
        "correct_en": "Indian railroad workers and merchants settling in East Africa.",
        "wrong_en_o1": "British citizens introducing Kenyans to baking and roasting.",
        "wrong_en_o2": "Yoruba traders assimilating into Kenyan markets.",
        "wrong_en_o3": "American settlers bringing over ideas of preservation and recycling.",
        "native_question": "Eponee likakiten inyamet a luipwaka kosokoni, ipyanikini iboro luipwaka lukakitijijim. Inyamata luh kijokis  itodikiti akapolok kajenaa kegwelet kalii?",
        "correct_native": "Itunga luka india luitenete ererwe  kiton lukebele luboyete okidee loka Africa",
        "wrong_native_o1": "Imusuguu  atodikini itung lukoo Kenya akiporee emugatii kakii wowa.",
        "wrong_native_o2": "Abunore kitunga kaluko Yoruba toma agwelet kokenya.",
        "wrong_native_o3": "Itunga luka amerika ayauni ekiponee kes lokaki gwaa  inyamat apak nepolo kakinyogokin akitosom",
        "swa_question": "Maandalizi na matumizi ya vyakula vingi vinavyopatikana masokoni yanajumuisha viungo na viungo kama vile bizari, manjano, giligilani, na tangawizi. Wasifu huu wa vyakula unaonyesha uhusiano gani wa kihistoria wa kibiashara?",
        "correct_swa": "Wafanyakazi wa reli na wafanyabiashara wa India wakiishi Afrika Mashariki.",
        "wrong_swa_o1": "Raia wa Uingereza wakiwafundisha Wakenya kuoka na kuchoma.",
        "wrong_swa_o2": "Wafanyabiashara wa Yoruba wakijiunga na masoko ya Kenya.",
        "wrong_swa_o3": "Walowezi wa Marekani wakileta mawazo ya kuhifadhi na kuchakata tena.",
    },
    {
        "ID": "custom_007",
        "Category": "Image 2 Questions",
        "file_id": "1x8AKBikOaDExx_LhB-vvuvLQZfuYBpeM",
        "eng_question": "Focusing on the tins in the image, what are they locally called?",
        "correct_en": "Gorogoro",
        "wrong_en_o1": "Kipimo",
        "wrong_en_o2": "Uzito",
        "wrong_en_o3": "Nafaka",
        "native_question": "Kiteyoo iboro nuejasii aputo ngolii, enyariti inyena?",
        "correct_native": "Agorogor",
        "wrong_native_o1": "Lokakitemio",
        "wrong_native_o2": "Alangir",
        "wrong_native_o3": "Iraito",
        "swa_question": "Kwa kuzingatia makopo kwenye picha, yanaitwaje hapa?",
        "correct_swa": "Gorogoro",
        "wrong_swa_o1": "Kipimo",
        "wrong_swa_o2": "Uzito",
        "wrong_swa_o3": "Nafaka",
    },
    {
        "ID": "custom_008",
        "Category": "Image 2 Questions",
        "file_id": "1x8AKBikOaDExx_LhB-vvuvLQZfuYBpeM",
        "eng_question": "The metal buckets and containers originally held things such as paint, seeds, and cooking fat. Now what are they used for? What does this demonstrate?",
        "correct_en": "Measurement and storage. Demonstrates recycling and repurposing.",
        "wrong_en_o1": "Preservation. Demonstrates innovative refrigeration and sustenance.",
        "wrong_en_o2": "Market display and branding. Demonstrates commerce.",
        "wrong_en_o3": "Making food portable. Demonstrates transportation and mobility.",
        "native_question": "Anddot loka chumat katon iboro,  itingito iboro bala  erangi,  ichoko katoni akinyet kedia. Kipokona iboru ngulu itosomayo eponali. Nyena itodikit.",
        "correct_native": "Akipima kakigwaa. Itodikini akinyogoo  akitosom iboro.",
        "wrong_native_o1": "Akisiboi boro. Itodikit achwaa nakitete nakkisiboi iboro.",
        "wrong_native_o2": "Akitodiar akigiriasha kiboro. Itodikini  egwelet.",
        "wrong_native_o3": "Akipyakin inyamat kobeikino adakar. Atodikin  akidak inyamat.",
        "swa_question": "Ndoo na vyombo vya chuma hapo awali vilikuwa na vitu kama vile rangi, mbegu, na mafuta ya kupikia. Sasa vinatumika kwa nini? Hii inaonyesha nini?",
        "correct_swa": "Vipimo na uhifadhi. Huonyesha kuchakata na kutumia tena.",
        "wrong_swa_o1": "Uhifadhi. Huonyesha uhifadhi na uhifadhi wa hali ya juu wa majokofu.",
        "wrong_swa_o2": "Maonyesho na chapa ya soko. Inaonyesha biashara.",
        "wrong_swa_o3": "Kutengeneza chakula kinachoweza kubebeka. Huonyesha usafiri na uhamaji.",
    },
    {
        "ID": "custom_009",
        "Category": "Image 3 Questions",
        "file_id": "1LagZEieb5SzUKwQ2jLfgGOXhjchC-_B0",
        "eng_question": "Why is this structure in the water?",
        "correct_en": "This structure used to be a restaurant a few years ago. Now, all that remains is the original steel structure standing in the water. This is in Lake Victoria and due to rising sea levels, flooding has cut more and more into the local land.",
        "wrong_en_o1": "This structure was built because birds kept flying onto the mainland, resting on people’s homes, and disturbing agricultural work.",
        "wrong_en_o2": "This structure is utilized for capturing fish. The net underneath traps seafood. Unfortunately, the design also attracts birds.",
        "wrong_en_o3": "The structure acts as a scarecrow. The birds scares away pests from meddling with important fishing areas for fishermen.",
        "native_question": "Nyo ejaree ibore eniii akipi?",
        "correct_native": "Ibore ngini namaa enyemere ikaru idis kau ngina. Kipokona ibore nideuna erait ichumai agwoete akipii. Nuh ejasii anam loka Victoria, atupa kakiyatakin kakipii, elamarosii akipii amaat kitungaa.",
        "wrong_native_o1": "Adukio ibore eni kanu aporiat ikweny, amaat, kakiboi kooria kitunga, kakishal asoma nukakor.",
        "wrong_native_o2": "Iboree ngini, itosomao akiiny ikolee. Enetii kopwap ekamii inyamt lukakipii. Aronus konye ekitenio ken enyarautii ikweny.",
        "wrong_native_o3": "Ibore kachan isomae bala Iboro lukarukwor. Ikwenye erukuorete  iboro llukarioko lu ebunete amunakin lukainyok naminyoto.",
        "swa_question": "Kwa nini muundo huu uko ndani ya maji?",
        "correct_swa": "Jengo hili lilikuwa mgahawa miaka michache iliyopita. Sasa, kilichobaki ni jengo la awali la chuma lililokuwa limesimama ndani ya maji. Hili liko katika Ziwa Victoria na kutokana na kupanda kwa kina cha bahari, mafuriko yamezidi kuathiri ardhi ya eneo hilo.",
        "wrong_swa_o1": "Jengo hili lilijengwa kwa sababu ndege waliendelea kuruka bara, wakipumzika kwenye nyumba za watu, na kusumbua kazi za kilimo.",
        "wrong_swa_o2": "Muundo huu hutumika kukamata samaki. Wavu ulio chini yake hunasa dagaa. Kwa bahati mbaya, muundo huo pia huvutia ndege.",
        "wrong_swa_o3": "Muundo huo hufanya kazi kama kitisho cha kunguru. Ndege hao huwaogopesha wadudu kutokana na kuingilia maeneo muhimu ya uvuvi kwa wavuvi.",
    },
    {
        "ID": "custom_010",
        "Category": "Image 3 Questions",
        "file_id": "1LagZEieb5SzUKwQ2jLfgGOXhjchC-_B0",
        "eng_question": "Which species of bird is shown by the black-headed bird with the white body in the top center of the image?",
        "correct_en": "African sacred ibis",
        "wrong_en_o1": "Great Egret",
        "wrong_en_o2": "Yellow-billed Stork",
        "wrong_en_o3": "Cormorant",
        "native_question": "Nyena ataker kikweny nitodikit ikweny nikironon kakou katon akwan  naka kwangan nejii kuju kiding kaputoo ngolii?",
        "correct_native": "Acuroit",
        "wrong_native_o1": "Amorocokin na aanywang",
        "wrong_native_o2": "Amorocokin na ejaatar keda eitoli lo eajany",
        "wrong_native_o3": "Ekeny lo lo ka akipi lo akirion",
        "swa_question": "Ni aina gani ya ndege inayoonyeshwa na ndege mwenye kichwa cheusi mwenye mwili mweupe katikati ya picha?",
        "correct_swa": "Ibis takatifu za Kiafrika",
        "wrong_swa_o1": "Mchuzi Mkuu",
        "wrong_swa_o2": "Stork mwenye mdomo wa manjano",
        "wrong_swa_o3": "Komoranti",
    },
    {
        "ID": "custom_011",
        "Category": "Image 3 Questions",
        "file_id": "1LagZEieb5SzUKwQ2jLfgGOXhjchC-_B0",
        "eng_question": "Which species of birds are all seen in the image?",
        "correct_en": "African sacred ibis, Great Egret, Yellow-billed Stork, Cormorant",
        "wrong_en_o1": "Grey Crowned Crane, African Jacana, Sooty Gull, Dwarf Bittern",
        "wrong_en_o2": "Little Grebe, Blacksmith Plover, Whiskered Tern, Hadada Ibis",
        "wrong_en_o3": "White-faced Whistling Duck, Red-knobbed Coot, Hamerkop, Striated Heron",
        "native_question": "Nyena atakere kikweny atakanete kaputo kana?",
        "correct_native": "Acuroit, Amorocokin na aanywang, Amorocokin na ejaatar keda eitoli lo eajany, Enyilil",
        "wrong_native_o1": "Ewalu, Ekeny lo lo ebeleketi amun, Ekeny lo lo ka anam lo ka asirikiny, Amorocokin na edit",
        "wrong_native_o2": "Ekeny lo lo eduki akipi, Ekeny lo lo egwedi apany, Ekeny lo lo ka akipi lo ejaatar keda ituben, Aruut",
        "wrong_native_o3": "Abanga na aanywang elosikinit kanyen, Ekeny lo lo ka akipi lo ejaatar keda arem na arengan kanyen, Among'ot, Amorocokin na korit",
        "swa_question": "Ni aina gani za ndege zinazoonekana kwenye picha?",
        "correct_swa": "Kwani mtakatifu wa Kiafrika, Mchuzi Mkuu, Korongo mwenye mdomo wa manjano, Cormorant",
        "wrong_swa_o1": "Korongo Mwenye Taji la Kijivu, Jacana wa Kiafrika, Shakwe Mdogo, Kibete",
        "wrong_swa_o2": "Kijidudu Kidogo, Mhunzi Plover, Mnyoofu Tern, Hadada Ibis",
        "wrong_swa_o3": "Bata Mzungu Mwenye Uso Mweupe, Coot Mwenye Vifundo Vikundu, Hamerkop, Korongo Mwenye Milia",
    },
    {
        "ID": "custom_012",
        "Category": "Image 4 Questions",
        "file_id": "1p6k5mfdOOWABJyA65VLM0TuoOPE7wF8I",
        "eng_question": "What is the most common use of the boats shown in the foreground?",
        "correct_en": "Fishing",
        "wrong_en_o1": "Leisure",
        "wrong_en_o2": "Sea travel",
        "wrong_en_o3": "Transporting goods",
        "native_question": "Ateter neetodunii aputoo nah , inyena chede nesikitsomate tunga",
        "correct_native": "Akiiny",
        "wrong_native_o1": "Aboyikini",
        "wrong_native_o2": "Akilosia Akaaree nakapolon",
        "wrong_native_o3": "Akitolosia Iboro",
        "swa_question": "Ni matumizi gani ya kawaida ya boti zinazoonyeshwa mbele?",
        "correct_swa": "Uvuvi",
        "wrong_swa_o1": "Burudani",
        "wrong_swa_o2": "Usafiri wa baharini",
        "wrong_swa_o3": "Kusafirisha bidhaa",
    },
    {
        "ID": "custom_013",
        "Category": "Image 4 Questions",
        "file_id": "1p6k5mfdOOWABJyA65VLM0TuoOPE7wF8I",
        "eng_question": "What body of water is being shown in the background?",
        "correct_en": "Lake Victoria",
        "wrong_en_o1": "The Indian Ocean",
        "wrong_en_o2": "The Red Sea",
        "wrong_en_o3": "The Great Lakes",
        "native_question": "Akipii nuh, luetodunii enyariti nyena.",
        "correct_native": "Anam loka Victoria",
        "wrong_native_o1": "Akare loka Hindi",
        "wrong_native_o2": "Anam napoleon lokared",
        "wrong_native_o3": "Anam nakapolon",
        "swa_question": "Ni maji gani yanayoonyeshwa chinichini?",
        "correct_swa": "Ziwa Victoria",
        "wrong_swa_o1": "Bahari ya Hindi",
        "wrong_swa_o2": "Bahari Nyekundu",
        "wrong_swa_o3": "Maziwa Makuu",
    },
    {
        "ID": "custom_014",
        "Category": "Image 4 Questions",
        "file_id": "1p6k5mfdOOWABJyA65VLM0TuoOPE7wF8I",
        "eng_question": "The woman in the front left is wearing a traditional Kenyan cloth. What is it called?",
        "correct_en": "Kitenge",
        "wrong_en_o1": "Kente",
        "wrong_en_o2": "Amauti",
        "wrong_en_o3": "Agbada",
        "native_question": "Aberu nigwoii, Kingaren kokedweny, enapit enaga lokagogong loka kenya. Enyariti enaga ngolii nyena?",
        "correct_native": "Kitenge",
        "wrong_native_o1": "Agbada",
        "wrong_native_o2": "Kente",
        "wrong_native_o3": "Amauti",
        "swa_question": "Mwanamke aliye mbele kushoto amevaa kitambaa cha kitamaduni cha Kenya. Kinaitwaje?",
        "correct_swa": "Kitenge",
        "wrong_swa_o1": "Kente",
        "wrong_swa_o2": "Amauti",
        "wrong_swa_o3": "Agbada",
    },
    {
        "ID": "custom_015",
        "Category": "Image 4 Questions",
        "file_id": "1p6k5mfdOOWABJyA65VLM0TuoOPE7wF8I",
        "eng_question": "Based on the text that can be found on the boats, where in Kenya is this photo taken?",
        "correct_en": "Dunga Beach",
        "wrong_en_o1": "Hippo Point",
        "wrong_en_o2": "Kit-Mikayi",
        "wrong_en_o3": "Impala Sanctuary",
        "native_question": "Kianyuyo agirit nejii ateter nejii aputoo ngolii, aputa nah keyaroo nama walii akwap loka Kenya?",
        "correct_native": "Dunga Beach",
        "wrong_native_o1": "Hippo Point",
        "wrong_native_o2": "Kit-Mikayi",
        "wrong_native_o3": "Impala Sanctuary",
        "swa_question": "Kulingana na maandishi yanayopatikana kwenye boti hizo, picha hii imepigwa wapi Kenya?",
        "correct_swa": "Ufuo wa Dunga",
        "wrong_swa_o1": "Kiboko Point",
        "wrong_swa_o2": "Kit-Mikayi",
        "wrong_swa_o3": "Hifadhi ya Impala",
    },
    {
        "ID": "custom_016",
        "Category": "Image 5 Questions",
        "file_id": "1sd9LqsAAbxvlXVRG7c0TgyO4lTf0dLHT",
        "eng_question": "How has M-PESA primarily transformed financial inclusion in Kenya?",
        "correct_en": "It has revolutionized Kenya’s financial inclusion as it allows people without bank accounts to securely deposit, withdraw, send, and receive money directly from their mobile phones.",
        "wrong_en_o1": "It has made the financial landscape of Kenya more exclusionary, as it is only offered to specific groups of people.",
        "wrong_en_o2": "It has made little to no difference, as most people don’t see a practical use for it.",
        "wrong_en_o3": "It has completely turned Kenya’s financial inclusion upside down, as it has led to mass inflation as a result of more cash being able to flow into and out of businesses.",
        "native_question": "Ikoni asoma ka mpessa ibelekini akijara kangotelei kokenya aii?",
        "correct_native": "Elonyaki eponee lokapeseii kokenya, Kanuu alachakii lumemmu kejasi kakaunt  abeikini  akigwaa, akiyakani,  kaki diyauni abolai alomuni kasiimu",
        "wrong_native_o1": "Edeuni aosma ken agelakina noi kanuu itsosomatete tunga liye ettiyakuna bon.",
        "wrong_native_o2": "Ayawu etanikina nedit, kanuu itete tunga luipwaka  kiteete adumunet kasoma ken",
        "wrong_native_o3": "Ebelok chut akiro kapesei kokenya kuju pwap, edipo apesei edachari pwap, nuh ebunete atupa kapesei akilom kalomar kogwelet.",
        "swa_question": "M-PESA imebadilisha vipi kimsingi ujumuishaji wa kifedha nchini Kenya?",
        "correct_swa": "Imebadilisha ujumuishaji wa kifedha wa Kenya kwani inaruhusu watu wasio na akaunti za benki kuweka, kutoa, kutuma, na kupokea pesa moja kwa moja kutoka kwa simu zao za mkononi kwa usalama.",
        "wrong_swa_o1": "Imefanya mazingira ya kifedha ya Kenya kuwa ya ubaguzi zaidi, kwani hutolewa kwa makundi maalum ya watu pekee.",
        "wrong_swa_o2": "Haijaleta tofauti kubwa au hakuna tofauti yoyote, kwani watu wengi hawaoni matumizi yake ya vitendo.",
        "wrong_swa_o3": "Imebadilisha kabisa ujumuishaji wa kifedha wa Kenya, kwani imesababisha mfumuko mkubwa wa bei kutokana na pesa nyingi zinazoweza kuingia na kutoka katika biashara.",
    },
    {
        "ID": "custom_017",
        "Category": "Image 5 Questions",
        "file_id": "1sd9LqsAAbxvlXVRG7c0TgyO4lTf0dLHT",
        "eng_question": "What are these local vendor stalls locally called?",
        "correct_en": "Kiosks",
        "wrong_en_o1": "Booths",
        "wrong_en_o2": "Newsstands",
        "wrong_en_o3": "Concessions",
        "native_question": "Nyena ekiror enyarite itogoi luchis luke bele nguluu kaduket?",
        "correct_native": "Kiosks",
        "wrong_native_o1": "Abutii",
        "wrong_native_o2": "Agwoete nakakirabaraba",
        "wrong_native_o3": "Eduka lochii.",
        "swa_question": "Vibanda hivi vya wauzaji wa ndani vinaitwaje?",
        "correct_swa": "Vibanda",
        "wrong_swa_o1": "Vibanda",
        "wrong_swa_o2": "Vibanda vya magazeti",
        "wrong_swa_o3": "Mapunguzo",
    },
    {
        "ID": "custom_018",
        "Category": "Image 5 Questions",
        "file_id": "1sd9LqsAAbxvlXVRG7c0TgyO4lTf0dLHT",
        "eng_question": "What is the significance of Coca-Cola being branded on the structure and in Kenya as a whole?",
        "correct_en": "Coca-Cola is the continent's largest bottler, with a bottling plant right in Kisumu. Coca-Cola provides stall owners with free or subsidized branded equipment like refrigerators, umbrellas, and painted storefronts. In exchange, the stall owners agree to stock Coca-Cola products and prominently display the logo.",
        "wrong_en_o1": "Coca-Cola has bought out all local markets in Kenya to create a monopoly over the soda market. They are advertising themselves on the structure to subconsciously make citizens want to buy their products more often.",
        "wrong_en_o2": "Coca-Cola sponsors local businesses and has incentives that pay people to openly wear, paint, and display their products. This has heavily stimulated the local markets but serves as a stark reminder that companies are slowly overtaking everyday life in Kenya.",
        "wrong_en_o3": "There is no significance behind the logo being painted; citizens around Kenya show love towards brands they love by painting them on their houses, cars, and businesses.",
        "native_question": "Nyena adumnute  ka cocacola ajaut kakigirashia Kotogo kangoli, katoni apwap nako Kenya kere.",
        "correct_native": "Ekampuni loka cocacal nesi le polo apwap  loka africa, ejasi kebele  lokapolon Kisumu. Ainakit ituga iboro luka long, bala elilinya, kakilony  echamuto itunga  egwelanar esoda kes, kakitodiar alama kes.",
        "wrong_native_o1": "Agwel cocala esokoni kisodai kijokis eteter erauni nesii loke bele kaniko kuju. Itodoarito  ebele kesi kotogoi kangulu teter itunga  egwelete iboro kes.",
        "wrong_native_o2": "Cocacola itopoloi ebele kesii koponee kaloka gwelakini itunga inagai, kakigir otogoi kes, kakitodiar iboro keso. Kakikono neni etopolo ebele loka kenya, itodikit ikampunio  epoloyete  atipet toma Okenya",
        "wrong_native_o3": "Emam adumunet kalama kangin, itung aloka keya elakarkina akipyakin alama ngini otogoi, amotokai kes kiton agwelet kes kalong",
        "swa_question": "Je, kuna umuhimu gani wa Coca-Cola kuwekewa chapa kwenye muundo huo na nchini Kenya kwa ujumla?",
        "correct_swa": "Coca-Cola ndiyo kiwanda kikubwa zaidi cha chupa barani Afrika, kikiwa na kiwanda cha kuwekea chupa huko Kisumu. Coca-Cola huwapa wamiliki wa vibanda vifaa vya bure au vya ruzuku kama vile jokofu, miavuli, na maduka yaliyopakwa rangi. Kwa kubadilishana, wamiliki wa vibanda hukubali kuhifadhi bidhaa za Coca-Cola na kuonyesha nembo hiyo waziwazi.",
        "wrong_swa_o1": "Coca-Cola imenunua masoko yote ya ndani nchini Kenya ili kuunda ukiritimba juu ya soko la soda. Wanajitangaza wenyewe kwenye muundo huo ili kuwafanya raia watake kununua bidhaa zao mara nyingi zaidi.",
        "wrong_swa_o2": "Coca-Cola inafadhili biashara za ndani na ina motisha zinazowalipa watu kuvaa, kupaka rangi, na kuonyesha bidhaa zao hadharani. Hii imechochea sana masoko ya ndani lakini inatumika kama ukumbusho dhahiri kwamba makampuni yanazidi polepole maisha ya kila siku nchini Kenya.",
        "wrong_swa_o3": "Hakuna umuhimu wowote nyuma ya nembo hiyo kuchorwa; raia kote Kenya huonyesha upendo kwa chapa wanazozipenda kwa kuzichora kwenye nyumba zao, magari, na biashara zao.",
    },
    {
        "ID": "custom_019",
        "Category": "Image 5 Questions",
        "file_id": "1sd9LqsAAbxvlXVRG7c0TgyO4lTf0dLHT",
        "eng_question": "What action is most likely to take place at a structure like this?",
        "correct_en": "Buying small produce and food.",
        "wrong_en_o1": "Selling old clothes for cash.",
        "wrong_en_o2": "Submit tax forms and documents.",
        "wrong_en_o3": "Setting up transportation to a place.",
        "native_question": "Nyena ibore nebeikini akisomakini kariri ketogo nejii aputo na?",
        "correct_native": "Agwele  iboro idis kinyamen",
        "wrong_native_o1": "Agwelanar inagai lumojong",
        "wrong_native_o2": "Ainakin  emusolo katoni apaplai",
        "wrong_native_o3": "Akipikin epone lokalosia namiche",
        "swa_question": "Ni hatua gani inayowezekana zaidi kufanyika katika jengo kama hili?",
        "correct_swa": "Kununua mazao madogo na chakula.",
        "wrong_swa_o1": "Kuuza nguo za zamani kwa pesa taslimu.",
        "wrong_swa_o2": "Wasilisha fomu na hati za kodi.",
        "wrong_swa_o3": "Kuweka usafiri hadi mahali.",
    },
    {
        "ID": "custom_020",
        "Category": "Image 6 Questions",
        "file_id": "1nATb92NkD0KfWqsiOsPCfeo9s-DXZB-m",
        "eng_question": "What rocks are shown in this image, and what is their significance?",
        "correct_en": "Calcium Stones from Kodiaga. These stones are traditionally consumed by pregnant women to supplement calcium, supporting bone health for both mother and child.",
        "wrong_en_o1": "Granite stones used for construction. Because granite is dense and durable, the pictured stones are used for heavy-duty construction and flooring.",
        "wrong_en_o2": "Limestone found from the shallow parts of Lake Victoria. Its softer texture makes it central to construction and agriculture. It is used for treating wastewater and fills in for food and toothpaste.",
        "wrong_en_o3": "Quarry stones from nearby mines. They are shaped because its raw state is often unusable. Frequently used for moisture resistance and underground construction",
        "native_question": "Nyena amoru nuitodute kaputo kana na nyena adumunte kes?",
        "correct_native": "Amoru nuka kalshiam nuko Kodiaga. Amoru nuh kakijara kanaka sem, anyemete aberu nepotiete  akiyatakini akwana kes ekalshiam, kanuka kingarakin akoyo kikoku katoken.",
        "wrong_native_o1": "Amoru nuka granait  nuitosomao aduko. Kanu alangiri egranait kibote de ding itosomao kesii aduko.",
        "wrong_native_o2": "Amoru  nuka laim, nue dummuno kotolima loka anam naka Victoria. Enonok naesi edipori itosomao kakoru kitoni akiduk. Itosomao akimada akipii kakitene ekeya kikyela.",
        "wrong_native_o3": "Amoru neelemute  kama ebokere emaram. Ekepite kanu emamm ketapana akitosom kesii kasodit. Duchu itosomao kanu emasiete akipii na adukio nuko pwap.",
        "swa_question": "Ni miamba gani inayoonyeshwa kwenye picha hii, na umuhimu wake ni upi?",
        "correct_swa": "Mawe ya Kalsiamu kutoka Kodiaga. Mawe haya huliwa kijadi na wanawake wajawazito ili kuongeza kalsiamu, na kusaidia afya ya mifupa kwa mama na mtoto.",
        "wrong_swa_o1": "Mawe ya granite yanayotumika kwa ajili ya ujenzi. Kwa sababu granite ni nzito na hudumu, mawe yaliyoonyeshwa kwenye picha hutumika kwa ajili ya ujenzi wa kazi nzito na sakafu.",
        "wrong_swa_o2": "Chokaa inayopatikana kutoka sehemu zisizo na kina kirefu za Ziwa Victoria. Umbile lake laini hulifanya kuwa muhimu kwa ujenzi na kilimo. Inatumika kwa ajili ya kutibu maji machafu na kujaza chakula na dawa ya meno.",
        "wrong_swa_o3": "Mawe ya machimbo kutoka migodi iliyo karibu. Yameumbwa kwa sababu hali yake mbichi mara nyingi haitumiki. Hutumika mara kwa mara kwa ajili ya upinzani wa unyevu na ujenzi wa chini ya ardhi",
    },
    {
        "ID": "custom_021",
        "Category": "Image 6 Questions",
        "file_id": "1nATb92NkD0KfWqsiOsPCfeo9s-DXZB-m",
        "eng_question": "What are the red balls in plastic and how were they prepared for the market?",
        "correct_en": "Groundnuts. They are prepped, soaked with hot water, and stirred and cooked over a heated pan. After this, the skin darkens and peels. They are cooled and they get crispy.",
        "wrong_en_o1": "Kidney beans. They are sorted and soaked. They are boiled for 10 minutes to denature the toxins. Eventually, the heat is reduced to a gentle simmer for under an hour until they are tender.",
        "wrong_en_o2": "Groundnuts. They are prepped and peeled. The water in a pot is seasoned as it is being boiled. When the water is boiled, the groundnuts are poured in and cooked for around three hours. After, they are soaked for a few hours so the salt penetrates the shell. They are drained and sold.",
        "wrong_en_o3": "Macadamia nuts. They are dehusked then the shells are cracked. They are dry roasted on a baking sheet for 10-12 minutes. During this process, they can be seasoned. They are stored and sold.",
        "native_question": "Nyena imoru luchusi lukaringak nwejasii okartasi, kosodi itenio kesii eponali?",
        "correct_native": "Emaide. Iwawo kakim nejasii kabalang, kowokoto.",
        "wrong_native_o1": "Amarage kepiro. Esekio kakitabaun kakipii.  Itukulauno kakipi idaikan itomon kanuka lemar adwarus. Kitolosi akiporee bala asaa epee kakim kadiosii kononokar",
        "wrong_native_o2": "Emaide. Epacho emaido. Akipii nujeasii amot nwemaka , etayatakino iboro lukajijim, kokulaete. Etolosit emadie akulau isaanii iuni, kedaunosii, utabauno isaan idisi, kolom abalang tenaa. Elemare akipii kijokis kagwelar.",
        "wrong_native_o3": "Amaide loka Macadamia. Epacho kaki pietar. Epeyoo kakim koaikan bala itomon karee. Apakii kanaa ipikino  abalang  kakipyakin ochwei, kagwelar.",
        "swa_question": "Mipira nyekundu iliyotengenezwa kwa plastiki ni nini na iliandaliwaje kwa ajili ya soko?",
        "correct_swa": "Karanga. Hutayarishwa, huloweshwa na maji ya moto, na kukorogwa na kupikwa kwenye sufuria yenye moto. Baada ya haya, ngozi hutiwa giza na kuchubuka. Hupozwa na kung'oka.",
        "wrong_swa_o1": "Maharage ya figo. Yamepangwa na kulowekwa. Yanachemshwa kwa dakika 10 ili kuondoa sumu. Hatimaye, moto hupunguzwa hadi kupikwa kidogo kwa chini ya saa moja hadi yawe laini.",
        "wrong_swa_o2": "Karanga. Hutayarishwa na kung'olewa. Maji kwenye sufuria hutiwa viungo yanapochemshwa. Maji yanapochemshwa, karanga hutiwa ndani na kupikwa kwa takriban saa tatu. Baada ya hapo, huloweshwa kwa saa chache ili chumvi iingie kwenye ganda. Huondolewa na kuuzwa.",
        "wrong_swa_o3": "Kokwa za makadamia. Huondolewa maganda kisha maganda hupasuka. Hukaushwa na kuokwa kwenye karatasi ya kuokea kwa dakika 10-12. Wakati wa mchakato huu, zinaweza kutiwa viungo. Huhifadhiwa na kuuzwa.",
    },
    {
        "ID": "custom_022",
        "Category": "Image 6 Questions",
        "file_id": "1nATb92NkD0KfWqsiOsPCfeo9s-DXZB-m",
        "eng_question": "What is in the white bucket?",
        "correct_en": "KSL Tropical Mints",
        "wrong_en_o1": "Glass marbles",
        "wrong_en_o2": "Commercially-produced blue stones",
        "wrong_en_o3": "Tanzanite gems",
        "native_question": "Nyena ejii andoot naka kwangan?",
        "correct_native": "Imints loka KSL",
        "wrong_native_o1": "Amoru luka glass",
        "wrong_native_o2": "Amoru nuka Tanzania",
        "wrong_native_o3": "Amoru nuitente nuka gwelanar.",
        "swa_question": "Ni nini kilichomo kwenye ndoo nyeupe?",
        "correct_swa": "KSL Mints za Tropiki",
        "wrong_swa_o1": "Marumaru za kioo",
        "wrong_swa_o2": "Mawe ya bluu yanayotengenezwa kibiashara",
        "wrong_swa_o3": "Vito vya Tanzanite",
    },
    {
        "ID": "custom_023",
        "Category": "Image 7 Questions",
        "file_id": "1jLX5cxvjljTk0PtMOaXFG5m055KSCptC",
        "eng_question": "What attire is being worn in this image?",
        "correct_en": "Owali sisal skirts",
        "wrong_en_o1": "Hula kahiko skirts",
        "wrong_en_o2": "Piupiu skirts",
        "wrong_en_o3": "Liku",
        "native_question": "Nyena iboro enapite kaputo kanaa?",
        "correct_native": "Asikat naka amujaj loka Owalii",
        "wrong_native_o1": "Asikat naka Hula kahiko",
        "wrong_native_o2": "Asikat naka piupiu",
        "wrong_native_o3": "Liku",
        "swa_question": "Ni mavazi gani yanayovaliwa katika picha hii?",
        "correct_swa": "Sketi za katani za Owali",
        "wrong_swa_o1": "Sketi za Hula kahiko",
        "wrong_swa_o2": "Sketi za Piupiu",
        "wrong_swa_o3": "Liku",
    },
    {
        "ID": "custom_024",
        "Category": "Image 7 Questions",
        "file_id": "1jLX5cxvjljTk0PtMOaXFG5m055KSCptC",
        "eng_question": "What community is this attire representative of?",
        "correct_en": "Luo",
        "wrong_en_o1": "Maasai",
        "wrong_en_o2": "Somali",
        "wrong_en_o3": "Meru",
        "native_question": "Nyena atakere eboikit anapa loka ibor kaluu?",
        "correct_native": "Dholuo",
        "wrong_native_o1": "Imaasai",
        "wrong_native_o2": "Isomalii",
        "wrong_native_o3": "Imeeru",
        "swa_question": "Mavazi haya yanawakilisha jamii gani?",
        "correct_swa": "Kiluo",
        "wrong_swa_o1": "Wamasai",
        "wrong_swa_o2": "Kisomali",
        "wrong_swa_o3": "Meru",
    },
    {
        "ID": "custom_025",
        "Category": "Image 7 Questions",
        "file_id": "1jLX5cxvjljTk0PtMOaXFG5m055KSCptC",
        "eng_question": "When and for what is this attire traditionally worn?",
        "correct_en": "By women and used for dancing. It is primarily worn during marriage ceremonies, celebrating newborns, and initiations. The loose fibers allow for free movement and for the skirt to sway and flow to the music.",
        "wrong_en_o1": "By children and used for dancing. It is primarily worn at children's birthday parties and celebrates the freedom of youth and celebration of life. Children dance in circles and play the nyatiti and abu",
        "wrong_en_o2": "By women and men and used for rituals. It is primarily worn during sacred moments for religion and coming of age. The community believes its material represents the creations of the Divine, and is thus holy attire.",
        "wrong_en_o3": "By women and used for fashion. It is primarily worn to show off the vibrant colors visible such as bright pink, purple, and teal. The stripping, softening, and dying fibers from the sisal plant make it very beautiful and this fine craftsmanship makes it an attire worn for fashionable and flashy purposes.",
        "native_question": "Worii kapaki kanii enapere iboroo iluu",
        "correct_native": "Enapete aberu  ileete. Enapio apakii nakaki numnum emanyit, kakidowuni ikoku. Anunuk  kamujajua elachankini itwanu akilej.",
        "wrong_native_o1": "Itosomate idwee akilejia, aoaki nakaki numunumu akidounio, katoni apak nakakitubio kaluka tumunak.",
        "wrong_native_o2": "Enapete aberu  ki kiliyok  esaa lokakinumnum. Itosomao apakii nakakinumnum iboro lukedeke katoni akinumnum amojong. Emunt ateker amuja egwoikit akiro kedeke, na eraiti akinap.",
        "wrong_native_o3": "Enapete aberu  kanukaminakes bon. Itosomao akitoduni irangino. Akitene ne jokuna nah idiport enapite iboro luh akitodiar amina kitunga kalakara.",
        "swa_question": "Mavazi haya huvaliwa lini na kwa ajili ya nini?",
        "correct_swa": "Na wanawake na hutumika kwa densi. Huvaliwa hasa wakati wa sherehe za ndoa, kusherehekea watoto wachanga, na unyago. Nyuzi hizo huru huruhusu harakati huru na sketi kutikisika na kutiririka kwa muziki.",
        "wrong_swa_o1": "Na watoto na hutumika kwa densi. Huvaliwa hasa kwenye sherehe za kuzaliwa kwa watoto na husherehekea uhuru wa ujana na sherehe ya maisha. Watoto hucheza kwenye miduara na hucheza nyatiti na abu",
        "wrong_swa_o2": "Na wanawake na wanaume na hutumika kwa ajili ya ibada. Huvaliwa hasa wakati wa nyakati takatifu kwa ajili ya dini na kukomaa. Jamii inaamini kwamba nyenzo zake zinawakilisha uumbaji wa Mungu, na hivyo ni mavazi matakatifu.",
        "wrong_swa_o3": "Na wanawake na hutumika kwa mitindo. Huvaliwa hasa kuonyesha rangi angavu zinazoonekana kama vile waridi angavu, zambarau, na bluu. Nyuzi zinazong'aa, zinazolainisha, na zinazokufa kutoka kwa mmea wa katani huifanya iwe nzuri sana na ufundi huu mzuri huifanya iwe vazi linalovaliwa kwa madhumuni ya mtindo na ya kuvutia.",
    },
    {
        "ID": "custom_026",
        "Category": "Image 8 Questions",
        "file_id": "1ZuWu-TWfzj_jE7c0Wgs8W3936X1Du-nQ",
        "eng_question": "What are the items held in the black tape or band?",
        "correct_en": "Natural luffa sponges",
        "wrong_en_o1": "White maize",
        "wrong_en_o2": "Dried plantains",
        "wrong_en_o3": "Sea sponges",
        "native_question": "Nyena ibor eruchokite kaunot kanakiryonon",
        "correct_native": "Echangwee Lokekitoi.",
        "wrong_native_o1": "Ekuridi lokekwangan",
        "wrong_native_o2": "Alaboro nukawoko.",
        "wrong_native_o3": "Echangwe loka anam nakapolon",
        "swa_question": "Ni vitu gani vilivyowekwa kwenye mkanda mweusi au utepe?",
        "correct_swa": "Sifongo za asili za luffa",
        "wrong_swa_o1": "Mahindi meupe",
        "wrong_swa_o2": "Ndizi zilizokaushwa",
        "wrong_swa_o3": "Sifongo za baharini",
    },
    {
        "ID": "custom_027",
        "Category": "Image 8 Questions",
        "file_id": "1ZuWu-TWfzj_jE7c0Wgs8W3936X1Du-nQ",
        "eng_question": "What is the purpose of the items held in the black tape or band?",
        "correct_en": "Body scrubbers. It is soaked in water and then they are perfect for exfoliation in the shower. They are also biodegradable and natural tools for scrubbing dishes and countertops.",
        "wrong_en_o1": "Consumption. It provides great nutrients such as potassium and vitamin B6. Eating them contributes to better gut health and thus acts as a healthy source of food for locals.",
        "wrong_en_o2": "Their natural liquid. They hold natural liquids that are mineral-rich and upon harvest are drained for these liquids. They are set out until all the liquid from the body is collected.",
        "wrong_en_o3": "Construction. They are torn apart and mixed with cement to fill in gaps in walls, floors, and roofs. It’s flexible and strong material makes it perfect for long-lasting structure enhancements and fixes.",
        "native_question": "Nyena atenikina kiboro erucjokite kaunot kankiryonon",
        "correct_native": "Echangwe Lokakiswaga akwan. Itabuono kakipi, Eboste tenan kalipo, katoni  iboro luiswagere ibor luenyamere.",
        "wrong_native_o1": "Luinyamat. Einakini iboro luedikte akwan  bala Potassium katoni Vitamin B6. Akinyam iboro ngulu einakini akok nejokuna, kosodi isomae bala inyamen nuelae kakwan.",
        "wrong_native_o2": "Akipii kes nwe tapana adumum. Itingito akipii nejasii kiminerals, edwenyo  elemunete akipii ngulu. Ejasii kosodii nelemenere akipii kijokis kakwan.",
        "wrong_native_o3": "Akiduk. Echilaro kokiding akinyalakin kesimit akilelebia aduyoka otogoi katoni kuju ketogo.  Anonnok kes idiporit itosomao adukia itogoi lu iboyete",
        "swa_question": "Madhumuni ya vitu vilivyoshikiliwa kwenye utepe mweusi au bendi ni nini?",
        "correct_swa": "Visafishaji vya mwili. Vinaloweshwa kwenye maji na kisha vinafaa kwa ajili ya kusugua ngozi wakati wa kuoga. Pia ni vifaa vinavyooza na vya asili vya kusugua vyombo na kaunta.",
        "wrong_swa_o1": "Matumizi. Hutoa virutubisho bora kama vile potasiamu na vitamini B6. Kula hivyo huchangia afya bora ya utumbo na hivyo hufanya kazi kama chanzo bora cha chakula kwa wenyeji.",
        "wrong_swa_o2": "Kimiminika chao cha asili. Huhifadhi vimiminika vya asili vyenye madini mengi na wakati wa mavuno huchujwa kwa ajili ya vimiminika hivi. Huwekwa nje hadi vimiminika vyote kutoka mwilini vitakapokusanywa.",
        "wrong_swa_o3": "Ujenzi. Hupasuka na kuchanganywa na saruji ili kujaza mapengo kwenye kuta, sakafu, na paa. Ni nyenzo inayonyumbulika na imara inayoifanya iwe bora kwa ajili ya uboreshaji na marekebisho ya muundo wa kudumu kwa muda mrefu.",
    },
    {
        "ID": "custom_028",
        "Category": "Image 8 Questions",
        "file_id": "1ZuWu-TWfzj_jE7c0Wgs8W3936X1Du-nQ",
        "eng_question": "What is the stack and on the right and what are the items used for?",
        "correct_en": "Raw jute fiber, which is utilized to make fabrics, bags, and mats. It can also be turned into yarn for weaving.",
        "wrong_en_o1": "Dried banana leaves. They have multiple uses, including being ground up for tea. They are also used to wrap up spices and small foods for preservation and and braided into ingata, which is used when carrying heavy things",
        "wrong_en_o2": "Mukeu, which is harvested to make strong, durable natural ropes. It is also used to weave baskets and traditional kiondo bags",
        "wrong_en_o3": "Stacks of locally made rubber bands. helpful for bundling to hold and bind products locally sold together. Also used for crafting, orthodontics, and transport.",
        "native_question": "Nyena iboro nu itogwoete nateten nyena asom kiboro kangulu?",
        "correct_native": "Anya nuka akorobos lu emameun apasao, lu eswamao nuka amunokin imanyas, ijokin, keda itatago. Ipedori bobo aenit keda airiit ecae nuka aodokin.",
        "wrong_native_o1": "Akwi nuka amuka na anyorire. Ejaatar keda aswamisinei ipu, naitisai aponia aitagulaun ecae lo amugit. Esubio bobo amunokin iasama keda aanyam na ititini aitogogong, keda aodokin arebeit (ingata) na ebeit aitit keda aiki iboro lu ititini.",
        "wrong_native_o2": "Mukeu, lo aponia eeswamao aoroi na agogong ka na itatene. Esubio bobo aodokin itatago keda itatago nuka kiondo.",
        "wrong_native_o3": "Itatago nuka apira nuka amukian aswam kotoma ateker. Konyout amunokin keda aitogogong keda amonikin emisago nuka amukian adukanat nepepe. Esubio bobo aitopol, acuran, keda aiki kwan.",
        "swa_question": "Mrundiko ni nini na upande wa kulia ni nini na vitu hivyo vinatumika kwa ajili ya nini?",
        "correct_swa": "Nyuzi mbichi za jute, ambazo hutumika kutengeneza vitambaa, mifuko, na mikeka. Pia zinaweza kubadilishwa kuwa uzi wa kufuma.",
        "wrong_swa_o1": "Majani ya ndizi yaliyokaushwa. Yana matumizi mengi, ikiwa ni pamoja na kusagwa kwa ajili ya chai. Pia hutumika kufunga viungo na vyakula vidogo kwa ajili ya kuhifadhi na kusokotwa kwenye ingata, ambayo hutumika wakati wa kubeba vitu vizito.",
        "wrong_swa_o2": "Mukeu, ambayo huvunwa ili kutengeneza kamba za asili zenye nguvu na za kudumu. Pia hutumika kusuka vikapu na mifuko ya kitamaduni ya kiondo",
        "wrong_swa_o3": "Mirundiko ya mikanda ya mpira iliyotengenezwa kienyeji. Inasaidia kwa kushikilia na kufunga bidhaa zinazouzwa kienyeji pamoja. Pia hutumika kwa ufundi, orthodontics, na usafiri.",
    },
    {
        "ID": "custom_029",
        "Category": "Image 9 Questions",
        "file_id": "1knmPao_ehrnkrqK_nOt5QRU1XAOAYaoL",
        "eng_question": "What type of fish is pictured?",
        "correct_en": "Omena (Silver Cyprinid)",
        "wrong_en_o1": "Tilapia (Ngege)",
        "wrong_en_o2": "Haplochromines (Fulu)",
        "wrong_en_o3": "Lake Magadi Tilapia (Oreochromis alcalicus)",
        "native_question": "Ani kanyara na ka igwenya na itodunitai kotoma epicha?",
        "correct_native": "Omena keda emukene",
        "wrong_native_o1": "Ngege keda ewape",
        "wrong_native_o2": "Fulu keda amuny",
        "wrong_native_o3": "Ngege ka anam lo Magadi Oreochromis alcalicus",
        "swa_question": "Ni aina gani ya samaki inayoonyeshwa kwenye picha?",
        "correct_swa": "Omena (Siprinidi ya Fedha)",
        "wrong_swa_o1": "Tilapia (Ngege)",
        "wrong_swa_o2": "Haplokromini (Fulu)",
        "wrong_swa_o3": "Ziwa Magadi Tilapia (Oreochromis alcalicus)",
    },
    {
        "ID": "custom_030",
        "Category": "Image 9 Questions",
        "file_id": "1knmPao_ehrnkrqK_nOt5QRU1XAOAYaoL",
        "eng_question": "What will this fish be used for after being sold?",
        "correct_en": "Food for humans and livestock feed",
        "wrong_en_o1": "Crushed up to be sold in local drinks",
        "wrong_en_o2": "Bait for game",
        "wrong_en_o3": "Sole cure for specific diseases",
        "native_question": "Ani aswam na ka igwenya na akaulo nuka amukit aijany?",
        "correct_native": "Aanyam nuka itunga keda aanyam nuka itisai ka ore",
        "wrong_native_o1": "Aanyam aitogogong aitit igwenya nuka amina",
        "wrong_native_o2": "Aitingis kede aiita itingisin nuka amukian apasao",
        "wrong_native_o3": "Aitabul keda amugit ecae lo amukian adukanat",
        "swa_question": "Samaki huyu atatumika kwa nini baada ya kuuzwa?",
        "correct_swa": "Chakula cha binadamu na mifugo",
        "wrong_swa_o1": "Imesagwa ili kuuzwa katika vinywaji vya kienyeji",
        "wrong_swa_o2": "Chambo cha mchezo",
        "wrong_swa_o3": "Tiba pekee kwa magonjwa maalum",
    },
    {
        "ID": "custom_031",
        "Category": "Image 9 Questions",
        "file_id": "1knmPao_ehrnkrqK_nOt5QRU1XAOAYaoL",
        "eng_question": "What does the processing method do?",
        "correct_en": "Concentrates nutrients, making them a dense source of essential dietary intake",
        "wrong_en_o1": "Gets rid of all germs and contaminants, lessening the chance of getting a disease due to intake",
        "wrong_en_o2": "Brings out the natural flavors making it more enjoyable for consumption",
        "wrong_en_o3": "Softens it making it easier to cook in dishes",
        "native_question": "Ani aswam na ka eipone lo lo ka aitoregeg boroke?",
        "correct_native": "Aenit keda aitukulun akula nuka aanyam, na itirini aitogogong aitit aswam na ka ekaulo ejok",
        "wrong_native_o1": "Aemwony ka aitorot kere imanyas keda agwelingot, aitiki itoroboto nuka aitun itingisin nuka amukian apasao",
        "wrong_native_o2": "Aitegulaun ecae lo amugit lo ka akwap aitogogong aitit anyam keda aiyalama",
        "wrong_native_o3": "Ainonok aitogogong aitit acurakin keda itugorisi lu amukian adukanat",
        "swa_question": "Mbinu ya usindikaji hufanya nini?",
        "correct_swa": "Huongeza virutubisho, na kuvifanya kuwa chanzo kingi cha ulaji muhimu wa lishe",
        "wrong_swa_o1": "Huondoa vijidudu na uchafu wote, na kupunguza uwezekano wa kupata ugonjwa kutokana na ulaji.",
        "wrong_swa_o2": "Hutoa ladha asilia na kuifanya iwe ya kufurahisha zaidi kwa matumizi",
        "wrong_swa_o3": "Hulainisha na kurahisisha kupika kwenye vyombo",
    },
    {
        "ID": "custom_032",
        "Category": "Image 9 Questions",
        "file_id": "1knmPao_ehrnkrqK_nOt5QRU1XAOAYaoL",
        "eng_question": "What is the woman pictured doing?",
        "correct_en": "Sorting small fish out on a mesh net to be dried in the sun for preservation",
        "wrong_en_o1": "Sorting small fish out on a mesh net to be put in a dish cooked for dinner",
        "wrong_en_o2": "Sorting small fish out on a mesh net to determine which are diseased or not",
        "wrong_en_o3": "Removing bones and skin off of harvested fish to be sold",
        "native_question": "Ani aswam na ka aberu na itodunitai kotoma epicha?",
        "correct_native": "Aitutun igwenya lukaditot kotoma egwang lo katenait aitwan kokiolong amunokin",
        "wrong_native_o1": "Aitutun igwenya lukaditot kotoma egwang lo katenait areit acurakin keda aanyam lo ka abwong",
        "wrong_native_o2": "Aitutun igwenya lukaditot kotoma egwang lo katenait aanyun lu ejaatar keda itingisin keda lu emameun",
        "wrong_native_o3": "Aitelekun akwito keda amun nuka igwenya lu aponia itelekun aitit amukit aijany",
        "swa_question": "Mwanamke aliyeonyeshwa kwenye picha anafanya nini?",
        "correct_swa": "Kuchagua samaki wadogo kwenye wavu wa matundu ili wakaushwe kwenye jua kwa ajili ya kuhifadhi",
        "wrong_swa_o1": "Kupanga samaki wadogo kwenye wavu wa matundu ili wawekwe kwenye sahani iliyopikwa kwa chakula cha jioni",
        "wrong_swa_o2": "Kuchagua samaki wadogo kwa kutumia wavu wa matundu ili kubaini ni samaki gani wenye ugonjwa au la",
        "wrong_swa_o3": "Kuondoa mifupa na ngozi ya samaki waliovunwa ili kuuzwa",
    },
    {
        "ID": "custom_033",
        "Category": "Image 10 Questions",
        "file_id": "1zah984l5GJlKEDbi8fF2qo9YSe4tspai",
        "eng_question": "What are these structures made out of?",
        "correct_en": "Framed with wood, sealed with a mixture containing cow dung. The roofs are also framed with timber, tightly packed with dried grass and reeds, and sealed with cow dung to prevent rain and precipitation wear and tear.",
        "wrong_en_o1": "Framed with wood, sealed with cement and wooden bricks. The roofs are also framed with timber, tightly packed with hay and bamboo, and sealed with cement to make it strong.",
        "wrong_en_o2": "The walls are strengthened with gypsum plaster. Corners are coated with lemon eucalyptus oil to repel insects. Small windows and openings for cooling ventilation. The roofs are also framed with timber and tightly packed with dried grass and reeds.",
        "wrong_en_o3": "Framed with sticks and packed with clay for a durable structure. The base in packed with grass and mud to make it comfortable to sleep. Hay is put on top of a roof frame made of bamboo.",
        "native_question": "Ani eipone lo eeswamao itogoi lu?",
        "correct_native": "Idulit keda akito, emwonyit keda anyonit nuka aite keda aoo. Eswi bobo idulit keda amukian, iwakit keda anya na anyorire keda aoroi, ka emwonyit keda anyonit nuka aite aiki ekisina ka akipi lo akwap aitorot.",
        "wrong_native_o1": "Idulit keda akito, emwonyit keda simeeti keda amukian nuka akito. Eswi bobo idulit keda amukian, iwakit keda anya na awoleto keda emuse, ka emwonyit keda simeeti aitogogong.",
        "wrong_native_o2": "Asirikitin itogogongit keda simeeti na ka anyonit. Imukian igwedi keda ekisina lo ka akito nuka akimait aiki ikurut. Itwolo na keditot keda ikomian nuka aiki ekuwam lo alilim. Eswi bobo idulit keda amukian ka iwakit keda anya na anyorire keda aoroi.",
        "wrong_native_o3": "Idulit keda aswaki ka iwakit keda anyonit nuka akorobos aponia aitogogong. Atwolo na koki iwakit keda anya keda anyonit na ebeit aiywun ejok aibo ka aitwong. Anya na awoleto ebiit kidama ka eswi lo amukian lo emuse.",
        "swa_question": "Miundo hii imetengenezwa kwa nini?",
        "correct_swa": "Imetengenezwa kwa mbao, imefungwa kwa mchanganyiko wenye kinyesi cha ng'ombe. Paa pia zimetengenezwa kwa mbao, zimefungwa kwa ukali na nyasi kavu na matete, na kufungwa kwa kinyesi cha ng'ombe ili kuzuia mvua na mvua kuchakaa.",
        "wrong_swa_o1": "Imetengenezwa kwa mbao, imefungwa kwa saruji na matofali ya mbao. Paa pia zimetengenezwa kwa mbao, zimefungwa kwa nyasi na mianzi vizuri, na kufungwa kwa saruji ili kuifanya iwe imara.",
        "wrong_swa_o2": "Kuta zimeimarishwa kwa plasta ya jasi. Pembe zimepakwa mafuta ya mikaratusi ya limau ili kufukuza wadudu. Madirisha madogo na nafasi za kupoeza hewa. Paa pia zimepambwa kwa mbao na zimefungwa vizuri kwa nyasi kavu na matete.",
        "wrong_swa_o3": "Imetengenezwa kwa vijiti na kufungwa kwa udongo kwa ajili ya muundo imara. Msingi wake umejaa nyasi na matope ili iwe rahisi kulala. Nyasi huwekwa juu ya fremu ya paa iliyotengenezwa kwa mianzi.",
    },
    {
        "ID": "custom_034",
        "Category": "Image 10 Questions",
        "file_id": "1zah984l5GJlKEDbi8fF2qo9YSe4tspai",
        "eng_question": "Who builds these structures in a Luo family?",
        "correct_en": "Both the male and female members of the family split the tasks",
        "wrong_en_o1": "The women in the family",
        "wrong_en_o2": "The eldest son",
        "wrong_en_o3": "The men in the family (the husband and sons)",
        "native_question": "Inai eduki itogoi lu kotoma ateker na ka Aluo?",
        "correct_native": "Imongin keda iaberu nuka ateker kere ejaatar keda aswamisinei aenit amorocokin",
        "wrong_native_o1": "Iaberu nuka ateker",
        "wrong_native_o2": "Okoki lo apol lo aberu na nasodit",
        "wrong_native_o3": "Imongin nuka ateker (okilen keda imongin lukaditot)",
        "swa_question": "Ni nani anayejenga majengo haya katika familia ya Waluo?",
        "correct_swa": "Wanaume na wanawake katika familia waligawanya majukumu",
        "wrong_swa_o1": "Wanawake katika familia",
        "wrong_swa_o2": "Mwana mkubwa",
        "wrong_swa_o3": "Wanaume katika familia (mume na wanawe)",
    },
    {
        "ID": "custom_035",
        "Category": "Image 10 Questions",
        "file_id": "1zah984l5GJlKEDbi8fF2qo9YSe4tspai",
        "eng_question": "Who lives in the largest hut?",
        "correct_en": "The first wife",
        "wrong_en_o1": "The husband",
        "wrong_en_o2": "The children",
        "wrong_en_o3": "Visiting guests",
        "native_question": "Inai eboie kotogo lo eponi lo ka ekaulo apol?",
        "correct_native": "Aberu na nasodit",
        "wrong_native_o1": "Okilen",
        "wrong_native_o2": "Imongin lukaditot",
        "wrong_native_o3": "Ipejon lu aponia bwan",
        "swa_question": "Nani anaishi katika kibanda kikubwa zaidi?",
        "correct_swa": "Mke wa kwanza",
        "wrong_swa_o1": "Mume",
        "wrong_swa_o2": "Watoto",
        "wrong_swa_o3": "Wageni wanaotembelea",
    },
    {
        "ID": "custom_036",
        "Category": "Image 10 Questions",
        "file_id": "1zah984l5GJlKEDbi8fF2qo9YSe4tspai",
        "eng_question": "Where was this image likely taken?",
        "correct_en": "Near Kit Mikayi",
        "wrong_en_o1": "Near Maasai Mara",
        "wrong_en_o2": "Near Mt. Kenya",
        "wrong_en_o3": "Near the Kenyan Coastal Region",
        "native_question": "Ani akwap na aponia itelekunai epicha lo?",
        "correct_native": "Diat Kit Mikayi",
        "wrong_native_o1": "Diat Maasai Mara",
        "wrong_native_o2": "Diat Mt. Kenya",
        "wrong_native_o3": "Diat anam lo ka akwap na a Kenya lo katenait kidama",
        "swa_question": "Huenda picha hii ilipigwa wapi?",
        "correct_swa": "Karibu na Kit Mikayi",
        "wrong_swa_o1": "Karibu na Maasai Mara",
        "wrong_swa_o2": "Karibu na Mlima Kenya",
        "wrong_swa_o3": "Karibu na Mkoa wa Pwani wa Kenya",
    },
    {
        "ID": "custom_037",
        "Category": "Image 10 Questions",
        "file_id": "1zah984l5GJlKEDbi8fF2qo9YSe4tspai",
        "eng_question": "What type of Kenyan hut does this image depict?",
        "correct_en": "Od Mikayi",
        "wrong_en_o1": "The Manyatta / Enkaji",
        "wrong_en_o2": "The Borana / Rendille Domes",
        "wrong_en_o3": "Swahili Coral Huts",
        "native_question": "Ani amaduli na ka a Kenya na itodunitai kotoma epicha?",
        "correct_native": "Od Mikayi",
        "wrong_native_o1": "Manyatta / Enkaji",
        "wrong_native_o2": "Borana / Rendille Domes",
        "wrong_native_o3": "Itogoi nuka Swahili Coral Huts",
        "swa_question": "Picha hii inaonyesha aina gani ya kibanda cha Kenya?",
        "correct_swa": "Od Mikayi",
        "wrong_swa_o1": "Manyatta / Enkaji",
        "wrong_swa_o2": "Mabawa ya Borana / Rendille",
        "wrong_swa_o3": "Vibanda vya Matumbawe vya Kiswahili",
    },
    {
        "ID": "custom_038",
        "Category": "Image 11 Questions",
        "file_id": "15A5HdRDwqnfbXayubjIrqsj5HR6dW0Kb",
        "eng_question": "Who’s clothes are seen on the lines?",
        "correct_en": "A family of multiple ages",
        "wrong_en_o1": "Just baby clothes",
        "wrong_en_o2": "Just women’s clothes",
        "wrong_en_o3": "Just men’s clothes",
        "native_question": "Ani iasama lu eanywunitai kidama ka aoroi?",
        "correct_native": "Lu ka ateker kere ka iisuben lu kapuaka",
        "wrong_native_o1": "Lu ka imongin lukaditot bon",
        "wrong_native_o2": "Lu ka iaberu bon",
        "wrong_native_o3": "Lu ka imongin bon",
        "swa_question": "Nguo za nani zinaonekana kwenye mstari?",
        "correct_swa": "Familia ya umri mbalimbali",
        "wrong_swa_o1": "Nguo za watoto tu",
        "wrong_swa_o2": "Nguo za wanawake pekee",
        "wrong_swa_o3": "Nguo za wanaume pekee",
    },
    {
        "ID": "custom_039",
        "Category": "Image 11 Questions",
        "file_id": "15A5HdRDwqnfbXayubjIrqsj5HR6dW0Kb",
        "eng_question": "What does this image say about the locals' treatment of clothes?",
        "correct_en": "They handwash it and hang it out on a clothesline",
        "wrong_en_o1": "They use washing and drying machines",
        "wrong_en_o2": "They sew their own clothes",
        "wrong_en_o3": "They collect clothes and often sell it for income",
        "native_question": "Ani itodunit epicha lo kotoma aswam na ka itunga ka ore keda iasama?",
        "correct_native": "Elwatar keda akanin ka aenit kidama ka aoroi lo ka aitwan kokiolong",
        "wrong_native_o1": "Emait iasama keda iaswamak lu ka ailwat keda aitwan",
        "wrong_native_o2": "Erodit iasama kwan iaswama nuka akanin",
        "wrong_native_o3": "Eiit iasama ka amukit aijany amonikin emisago",
        "swa_question": "Picha hii inasema nini kuhusu jinsi wenyeji wanavyoshughulikia nguo?",
        "correct_swa": "Wanaiosha kwa mikono na kuitundika kwenye kamba ya nguo",
        "wrong_swa_o1": "Wanatumia mashine za kufulia na kukaushia",
        "wrong_swa_o2": "Wanashona nguo zao wenyewe",
        "wrong_swa_o3": "Wanakusanya nguo na mara nyingi huziuza kwa ajili ya mapato",
    },
    {
        "ID": "custom_040",
        "Category": "Image 11 Questions",
        "file_id": "15A5HdRDwqnfbXayubjIrqsj5HR6dW0Kb",
        "eng_question": "What are the animals seen below the clothesline and what is their purpose?",
        "correct_en": "They are the Muscovy duck and are often kept by farmers and backyard poultry keepers. They are quiet birds used for their meat, eggs, and natural pest control.",
        "wrong_en_o1": "They are Khaki Campbell and wander from the nearby Lake Victoria into the villagers backyard to feast on insects in the grass.",
        "wrong_en_o2": "They are chickens and are often used for agricultural purposes. They provide eggs to use in recipes and eat for nutrients, and are often sold or harvested by farmers for their meat.",
        "wrong_en_o3": "They are an invasive duck species that eats surrounding vegetation and insects.",
        "native_question": "Ani itisai lu eanywunitai koki ka aoroi lo ka iasama, ka ani aswamikebe?",
        "correct_native": "Irai abangat na ka Muscovy na eeswamao nuka itunga ka ore keda amonikin iasama. Irai ikwenta lu emameun iisuben lu eeswamao nuka aanyam, akwito, keda aiki ikurut lu aponia bwan.",
        "wrong_native_o1": "Irai ikoko lu eeswamao nuka aswam na ka akorobos. Emonikit akwito nuka aswam kotoma acurakin keda aanyam, ka amukit aijany nuka amonikin emisago.",
        "wrong_native_o2": "Irai abangat na ka Khaki Campbell na elosit katenait anam lo ka Victoria aitun ikurut kotoma anya nuka itunga ka ore.",
        "wrong_native_o3": "Irai abangat na ka areit na anyarait na aponia bwan na aanyam anya keda ikurut lu ka akwap.",
        "swa_question": "Ni wanyama gani wanaoonekana chini ya kamba ya nguo na madhumuni yao ni nini?",
        "correct_swa": "Wao ni bata aina ya Muscovy na mara nyingi hufugwa na wakulima na wafugaji wa kuku wa mashambani. Ni ndege tulivu wanaotumika kwa nyama yao, mayai, na udhibiti wa wadudu waharibifu wa asili.",
        "wrong_swa_o1": "Wao ni Khaki Campbell na hutangatanga kutoka Ziwa Victoria lililo karibu na kuingia kwenye uwanja wa nyuma wa wanakijiji ili kula wadudu kwenye nyasi.",
        "wrong_swa_o2": "Ni kuku na mara nyingi hutumika kwa madhumuni ya kilimo. Hutoa mayai ya kutumia katika mapishi na kula kwa ajili ya virutubisho, na mara nyingi huuzwa au kuvunwa na wakulima kwa ajili ya nyama yao.",
        "wrong_swa_o3": "Ni aina ya bata vamizi wanaokula mimea na wadudu wanaowazunguka.",
    },
    {
        "ID": "custom_041",
        "Category": "Image 12 Questions",
        "file_id": "1UlOl57GPNS9cr8MoyA_xCKfbCvNVRPve",
        "eng_question": "What is this man doing?",
        "correct_en": "Hammering steel onto a boat to build it",
        "wrong_en_o1": "Repairing a wooden boat",
        "wrong_en_o2": "Preparing to launch a boat into the water.",
        "wrong_en_o3": "Building a long wooden crate",
        "native_question": "Ani aswam na ka okilen lo itodunitai kotoma epicha?",
        "correct_native": "Eiramit apira kidama ka atatago na ka atuboi aduko",
        "wrong_native_o1": "Ainyasum atuboi na ka akito na abila",
        "wrong_native_o2": "Areit aiki atuboi kotoma akipi",
        "wrong_native_o3": "Aduko atatago na ka akito na awoleto",
        "swa_question": "Mwanamume huyu anafanya nini?",
        "correct_swa": "Kupiga chuma kwenye mashua ili kuijenga",
        "wrong_swa_o1": "Kutengeneza mashua ya mbao",
        "wrong_swa_o2": "Kujiandaa kuzindua mashua ndani ya maji.",
        "wrong_swa_o3": "Kujenga kreti ndefu ya mbao",
    },
    {
        "ID": "custom_042",
        "Category": "Image 12 Questions",
        "file_id": "1UlOl57GPNS9cr8MoyA_xCKfbCvNVRPve",
        "eng_question": "This task will most likely take in total…",
        "correct_en": "A few weeks to a few months",
        "wrong_en_o1": "A day",
        "wrong_en_o2": "A few days",
        "wrong_en_o3": "A few years",
        "native_question": "Aswam na, kotoma omarisin kere, elosit aitun...",
        "correct_native": "Iiapas lu keditot aitun iapas lu kapuaka",
        "wrong_native_o1": "Aparan iasama nepepe",
        "wrong_native_o2": "Aparanin lu keditot bon",
        "wrong_native_o3": "Ikarin lu keditot bon",
        "swa_question": "Kazi hii ina uwezekano mkubwa wa kuchukua jumla…",
        "correct_swa": "Wiki chache hadi miezi michache",
        "wrong_swa_o1": "Siku moja",
        "wrong_swa_o2": "Siku chache",
        "wrong_swa_o3": "Miaka michache",
    },
    {
        "ID": "custom_043",
        "Category": "Image 12 Questions",
        "file_id": "1UlOl57GPNS9cr8MoyA_xCKfbCvNVRPve",
        "eng_question": "This man’s job is",
        "correct_en": "A boatmaker",
        "wrong_en_o1": "An engineer",
        "wrong_en_o2": "A fisherman",
        "wrong_en_o3": "A welder",
        "native_question": "Ani aswam na ka okilen lo?",
        "correct_native": "Ekaaswaman lo ka atuboin",
        "wrong_native_o1": "Einjinia lo ka akwap",
        "wrong_native_o2": "Ekaalwaran lo ka igwenya",
        "wrong_native_o3": "Ekaaswaman lo ka apira",
        "swa_question": "Kazi ya mtu huyu ni",
        "correct_swa": "Mtengenezaji wa mashua",
        "wrong_swa_o1": "Mhandisi",
        "wrong_swa_o2": "Mvuvi",
        "wrong_swa_o3": "Mshonaji",
    },
    {
        "ID": "custom_044",
        "Category": "Image 13 Questions",
        "file_id": "1OpSdDSjVmev0-RXs18Y9_dnmEzJBzFMj",
        "eng_question": "This image shows the most common means of transportation are…",
        "correct_en": "Boda-bodas, tuktuks, cars",
        "wrong_en_o1": "Motorcycles, minibuses, railways",
        "wrong_en_o2": "Bikes and motorcycles",
        "wrong_en_o3": "Cars, motorcycles, ferries",
        "native_question": "Epicha lo itodunit eipone lo lo ka aiki lo apol elosit katenait...",
        "correct_native": "Boda-boda, tuktuk, keda emuuga",
        "wrong_native_o1": "Pikipiki, matatu, keda egari lo ka apira lo agogong",
        "wrong_native_o2": "Ibasikeli keda pikipiki",
        "wrong_native_o3": "Emuuga, pikipiki, keda atuboi na apol na ka akipi",
        "swa_question": "Picha hii inaonyesha njia za kawaida za usafiri ni…",
        "correct_swa": "Boda-boda, tuktuks, magari",
        "wrong_swa_o1": "Pikipiki, mabasi madogo, reli",
        "wrong_swa_o2": "Baiskeli na pikipiki",
        "wrong_swa_o3": "Magari, pikipiki, vivuko",
    },
    {
        "ID": "custom_045",
        "Category": "Image 13 Questions",
        "file_id": "1OpSdDSjVmev0-RXs18Y9_dnmEzJBzFMj",
        "eng_question": "What are the motorcycle taxis in this image called?",
        "correct_en": "Boda-bodas",
        "wrong_en_o1": "Autobikes",
        "wrong_en_o2": "Okada",
        "wrong_en_o3": "Cruisers",
        "native_question": "Ani eboiet nuka pikipiki nuka itolosi itunga kotoma epicha lo?",
        "correct_native": "Boda-boda",
        "wrong_native_o1": "Autobikes",
        "wrong_native_o2": "Okada",
        "wrong_native_o3": "Cruisers",
        "swa_question": "Teksi za pikipiki katika picha hii zinaitwaje?",
        "correct_swa": "Boda-boda",
        "wrong_swa_o1": "Baiskeli za magari",
        "wrong_swa_o2": "Okada",
        "wrong_swa_o3": "Mabaharia",
    },
    {
        "ID": "custom_046",
        "Category": "Image 13 Questions",
        "file_id": "1OpSdDSjVmev0-RXs18Y9_dnmEzJBzFMj",
        "eng_question": "Which is a likely popular motorcycle model in this area?",
        "correct_en": "Honda CG125",
        "wrong_en_o1": "Ducati",
        "wrong_en_o2": "BMW Motorrad",
        "wrong_en_o3": "Honda NC750X",
        "native_question": "Ani eipone lo ka pikipiki lo amina ka lo apol aswam kotoma aibo na?",
        "correct_native": "Honda CG125",
        "wrong_native_o1": "Ducati",
        "wrong_native_o2": "BMW Motorrad",
        "wrong_native_o3": "Honda NC750X",
        "swa_question": "Ni pikipiki gani inayoweza kuwa maarufu katika eneo hili?",
        "correct_swa": "Honda CG125",
        "wrong_swa_o1": "Ducati",
        "wrong_swa_o2": "BMW Motorrad",
        "wrong_swa_o3": "Honda NC750X",
    },
    {
        "ID": "custom_047",
        "Category": "Image 13 Questions",
        "file_id": "1OpSdDSjVmev0-RXs18Y9_dnmEzJBzFMj",
        "eng_question": "Delivery drivers most frequently drive",
        "correct_en": "Motorcycles or bikes",
        "wrong_en_o1": "Cars",
        "wrong_en_o2": "Trucks",
        "wrong_en_o3": "Tuktuks",
        "native_question": "Ekaaiton ka ekaatwan lo ka iasama katenait amukit aijany itoriti elosi aiki...",
        "correct_native": "Pikipiki keda ibasikeli",
        "wrong_native_o1": "Emuuga",
        "wrong_native_o2": "Emuuga lu apolok lu ka aiki iboro",
        "wrong_native_o3": "Tuktuk",
        "swa_question": "Madereva wa usafirishaji mara nyingi huendesha gari",
        "correct_swa": "Pikipiki au pikipiki",
        "wrong_swa_o1": "Magari",
        "wrong_swa_o2": "Malori",
        "wrong_swa_o3": "Tuktuk",
    },
    {
        "ID": "custom_048",
        "Category": "Image 14 Questions",
        "file_id": "15fdFEbGyGGQH5565RDtqjSPXCXWcIbyF",
        "eng_question": "This image was taken at a…",
        "correct_en": "Gas station",
        "wrong_en_o1": "Restaurant",
        "wrong_en_o2": "Mechanic stop/auto repair",
        "wrong_en_o3": "Water filling station",
        "native_question": "Epicha lo aponia itelekunai kotoma...",
        "correct_native": "Aibo na ka amait amauta nuka emuuga",
        "wrong_native_o1": "Aibo na ka aanyam",
        "wrong_native_o2": "Aibo na ka ainyasum emuuga",
        "wrong_native_o3": "Aibo na ka amait akipi kotoma icupa",
        "swa_question": "Picha hii ilipigwa katika…",
        "correct_swa": "Kituo cha mafuta",
        "wrong_swa_o1": "Mkahawa",
        "wrong_swa_o2": "Urekebishaji wa kituo/magari ya fundi",
        "wrong_swa_o3": "Kituo cha kujaza maji",
    },
    {
        "ID": "custom_049",
        "Category": "Image 14 Questions",
        "file_id": "15fdFEbGyGGQH5565RDtqjSPXCXWcIbyF",
        "eng_question": "What is in the bottle on the shelf?",
        "correct_en": "Soap / Multipurpose Cleaner",
        "wrong_en_o1": "Laundry detergent",
        "wrong_en_o2": "Soda",
        "wrong_en_o3": "Water",
        "native_question": "Ani ejaatatar kotoma ecupa kidama ka emesa lo ka aitwong?",
        "correct_native": "Aboslo na ka ailwat iboro kere",
        "wrong_native_o1": "Aboslo na ka ailwat iasama ka ore",
        "wrong_native_o2": "Soda lo amugit",
        "wrong_native_o3": "Akipi",
        "swa_question": "Ni nini kilichomo kwenye chupa kwenye rafu?",
        "correct_swa": "Sabuni / Kisafishaji cha Matumizi Mengi",
        "wrong_swa_o1": "Sabuni ya kufulia",
        "wrong_swa_o2": "Soda",
        "wrong_swa_o3": "Maji",
    },
    {
        "ID": "custom_050",
        "Category": "Image 14 Questions",
        "file_id": "15fdFEbGyGGQH5565RDtqjSPXCXWcIbyF",
        "eng_question": "What is the company featured on the red shirt?",
        "correct_en": "National Oil Corporation of Kenya",
        "wrong_en_o1": "Rubis Energy Kenya",
        "wrong_en_o2": "KFC",
        "wrong_en_o3": "Safaricom",
        "native_question": "Ani aodokin na ka ekaaswaman lo ejaatar keda eoroi lo arengan?",
        "correct_native": "National Oil Corporation of Kenya",
        "wrong_native_o1": "Rubis Energy Kenya",
        "wrong_native_o2": "KFC",
        "wrong_native_o3": "Safaricom",
        "swa_question": "Ni kampuni gani iliyoonyeshwa kwenye shati jekundu?",
        "correct_swa": "Shirika la Kitaifa la Mafuta la Kenya",
        "wrong_swa_o1": "Nishati ya Rubis Kenya",
        "wrong_swa_o2": "KFC",
        "wrong_swa_o3": "Safaricom",
    },
    {
        "ID": "custom_051",
        "Category": "Image 14 Questions",
        "file_id": "15fdFEbGyGGQH5565RDtqjSPXCXWcIbyF",
        "eng_question": "What is this person’s job?",
        "correct_en": "Dispensing fuel and processing payments",
        "wrong_en_o1": "Serving food to waiting customers",
        "wrong_en_o2": "Driving taxis and being a mechanic",
        "wrong_en_o3": "Cleaning up and being a janitor",
        "native_question": "Ani aswam na ka itunganan lo?",
        "correct_native": "Amait amauta nuka emuuga keda amunokin emisago nuka amukit aijany",
        "wrong_native_o1": "Aiki aanyam katenait itunga lu eeswamao",
        "wrong_native_o2": "Aiki emuuga keda aswam na ka ainyasum iboro lu abila",
        "wrong_native_o3": "Ailwat akwap keda aswam na ka aiki akorobos kere",
        "swa_question": "Kazi ya mtu huyu ni nini?",
        "correct_swa": "Kutoa mafuta na malipo ya usindikaji",
        "wrong_swa_o1": "Kuhudumia wateja wanaosubiri chakula",
        "wrong_swa_o2": "Kuendesha teksi na kuwa fundi",
        "wrong_swa_o3": "Kusafisha na kuwa mlinzi wa nyumba",
    },
]

def load_drive_image(file_id, _cache={}):
    """Download a Google Drive image by file ID, with in-memory caching."""
    if file_id in _cache:
        return _cache[file_id]
    url = f"https://drive.google.com/uc?export=download&id={file_id}"
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        img = Image.open(io.BytesIO(r.content)).convert("RGB")
        _cache[file_id] = img
        return img
    except Exception as e:
        print(f"  Warning: could not load image {file_id}: {e}")
        return Image.new("RGB", (224, 224), color="gray")

# Maps each language to its question/answer field names
LANGUAGE_CONFIGS = {
    "English": {
        "question_key": "eng_question",
        "correct_key":  "correct_en",
        "wrong_keys":   ["wrong_en_o1", "wrong_en_o2", "wrong_en_o3"],
    },
    "Ateso": {
        "question_key": "native_question",
        "correct_key":  "correct_native",
        "wrong_keys":   ["wrong_native_o1", "wrong_native_o2", "wrong_native_o3"],
    },
    "Swahili": {
        "question_key": "swa_question",
        "correct_key":  "correct_swa",
        "wrong_keys":   ["wrong_swa_o1", "wrong_swa_o2", "wrong_swa_o3"],
    },
}

print(f"Dataset loaded: {len(CUSTOM_DATA)} questions x 3 languages = {len(CUSTOM_DATA)*3} total evaluations.")


### 6. Preview Dataset

In [ ]:
# ============================================================
# STEP 5: Preview Dataset — First 2 Samples Across All Languages
# ============================================================
for i, sample in enumerate(CUSTOM_DATA[:2]):
    print(f"--- Sample {i+1} ({sample['Category']}) ---")
    for lang, cfg in LANGUAGE_CONFIGS.items():
        print(f"  [{lang}] Q: {sample[cfg['question_key']][:80]}")
        print(f"         A: {sample[cfg['correct_key']][:80]}")
    print()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for i, sample in enumerate(CUSTOM_DATA[:2]):
    img = load_drive_image(sample["file_id"])
    axes[i].imshow(img)
    axes[i].set_title(f"Sample {i+1}\n{sample['Category'][:20]}", fontsize=8)
    axes[i].axis("off")
plt.suptitle("Custom Kenyan Dataset — First 2 Samples", fontsize=11)
plt.tight_layout()
plt.show()


### 7. Helper Functions

In [ ]:
# ==========================================================
# STEP 6: Define Helper Functions
# ===========================================================

def build_mcqa_prompt(question, choices):
    """
    Format question + choices into a multiple choice prompt.
    Instructs the model to respond with only A, B, C, or D.
    """
    prompt = (
        "You are answering a visual multiple choice question. "
        "You must respond with ONLY a single letter: A, B, C, or D.\n\n"
        "Example:\n"
        "Question: What color is the sky?\n"
        "A. Red\nB. Blue\nC. Green\nD. Yellow\n"
        "Answer: B\n\n"
        "Now answer this question:\n"
        f"Question: {question}\n"
    )
    for label, choice in zip(["A", "B", "C", "D"], choices):
        prompt += f"{label}. {choice}\n"
    prompt += "Answer:"
    return prompt


def extract_predicted_label(predicted_text, choices):
    """
    Extract predicted label (A/B/C/D) from model output.
    Strategy 1: look for A/B/C/D letter in model output.
    Strategy 2: if no letter found, match answer text directly.
    """
    for label in ["A", "B", "C", "D"]:
        if label in predicted_text.upper():
            return label
    for label, choice in zip(["A", "B", "C", "D"], choices):
        if choice.lower() in predicted_text.lower():
            return label
    return None


### 8. Phi-3 Vision Inference Function

In [ ]:
# ==========================================================
# STEP 7: Phi-3 Vision Inference Function
# ===========================================================

def run_phi3(image, prompt, processor, model):
    """
    Run Phi-3 Vision 128K inference on an image + text prompt.
    """
    messages = [
        {
            "role": "user",
            "content": "<|image_1|>\n" + prompt,
        }
    ]

    text = processor.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = processor(text=text, images=[image], return_tensors="pt").to(
        model.device, torch.bfloat16
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
        )

    input_len = inputs["input_ids"].shape[-1]
    predicted_text = processor.tokenizer.decode(
        outputs[0][input_len:], skip_special_tokens=True
    ).strip()

    return predicted_text


### 9. Evaluation Function

In [ ]:
# ============================================================
# STEP 8: Define Evaluation Function
# ============================================================

def evaluate_language(data, lang, cfg, processor, model):
    """Evaluate Phi-3 Vision on one language slice of the dataset."""
    question_key = cfg["question_key"]
    correct_key  = cfg["correct_key"]
    wrong_keys   = cfg["wrong_keys"]

    correct_count = 0
    predictions   = []
    total         = len(data)

    print(f"\nRunning evaluation on [{lang}] — {total} samples...")
    print("-" * 40)

    for i, sample in enumerate(data):
        if (i + 1) % 10 == 0 or i == 0:
            print(f"  Processing sample {i+1}/{total}...")

        question    = sample[question_key]
        correct_ans = sample[correct_key]
        choices     = [correct_ans] + [sample[k] for k in wrong_keys]
        random.shuffle(choices)
        correct_label = ["A", "B", "C", "D"][choices.index(correct_ans)]

        prompt = build_mcqa_prompt(question, choices)

        try:
            image           = load_drive_image(sample["file_id"])
            predicted_text  = run_phi3(image, prompt, processor, model)
            predicted_label = extract_predicted_label(predicted_text, choices)
            is_correct      = (predicted_label == correct_label)
            if is_correct:
                correct_count += 1
        except Exception as e:
            print(f"  Error on sample {i+1}: {e}")
            predicted_text  = "ERROR"
            predicted_label = None
            is_correct      = False

        predictions.append({
            "sample_id":       sample["ID"],
            "category":        sample["Category"],
            "language":        lang,
            "question":        question,
            "choices":         {l: c for l, c in zip(["A","B","C","D"], choices)},
            "correct_answer":  correct_ans,
            "correct_label":   correct_label,
            "predicted_text":  predicted_text,
            "predicted_label": predicted_label,
            "is_correct":      is_correct,
        })

    accuracy = (correct_count / total) * 100 if total > 0 else 0
    return accuracy, predictions


### 10. Sanity Check (5 English Samples)

In [ ]:
# ============================================================
# STEP 9: Sanity Check on First 5 Samples (English Only)
# ============================================================
print("\nRunning sanity check on 5 English samples...")

_, sanity_predictions = evaluate_language(
    CUSTOM_DATA[:5], "English", LANGUAGE_CONFIGS["English"], processor, model
)

print("\nSanity check results:")
for i, pred in enumerate(sanity_predictions, 1):
    status = "✓" if pred["is_correct"] else "✗"
    print(f"  [{status}] Q{i}: {pred['question'][:80]}")
    print(f"        Predicted: {pred['predicted_label']} "
          f"| Correct: {pred['correct_label']} "
          f"| Model output: {pred['predicted_text']}")

print("\nIf results look reasonable, proceed to full evaluation...\n")


### 11. Full Evaluation — All 3 Languages

In [ ]:
# ============================================================
# STEP 10: Full Evaluation — All 3 Languages (50 questions each)
# ============================================================
all_results = {}

for lang, cfg in LANGUAGE_CONFIGS.items():
    acc, preds = evaluate_language(CUSTOM_DATA, lang, cfg, processor, model)
    all_results[lang] = {"accuracy": acc, "predictions": preds}
    print(f"  [{lang}] Accuracy: {acc:.2f}%")


### 12. Results

In [ ]:
# ============================================================
# STEP 11: Print Final Results
# ============================================================
print("\n" + "="*60)
print(f"  Final Results — Phi-3 Vision 128K | Kenyan Culture Dataset")
print("="*60)

for lang, res in all_results.items():
    preds = res["predictions"]
    acc   = res["accuracy"]

    category_scores = {}
    for pred in preds:
        cat = pred["category"]
        if cat not in category_scores:
            category_scores[cat] = {"correct": 0, "total": 0}
        category_scores[cat]["total"] += 1
        if pred["is_correct"]:
            category_scores[cat]["correct"] += 1

    print(f"\n── {lang} ──")
    print("Per-image accuracy:")
    for cat, scores in sorted(category_scores.items()):
        cat_acc = (scores["correct"] / scores["total"]) * 100
        print(f"  {cat:<45} {cat_acc:.1f}%  ({scores['correct']}/{scores['total']})")
    print(f"  Overall: {acc:.2f}%  |  Correct: {sum(p['is_correct'] for p in preds)}/{len(preds)}")

print(f"\nRandom chance baseline: 25.00% (4 choices)")
print("="*60)


### 13. Visualise Results

In [ ]:
# ============================================================
# STEP 12: Visualise Results
# ============================================================

# Bar chart — accuracy across all 3 languages
langs  = list(all_results.keys())
accs   = [all_results[l]["accuracy"] for l in langs]
colors = ["#4C72B0", "#DD8452", "#55A868"]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(langs, accs, color=colors, width=0.5)
ax.axhline(25, color="gray", linestyle="--", linewidth=1, label="Random baseline (25%)")
ax.set_ylim(0, 100)
ax.set_ylabel("Accuracy (%)")
ax.set_title("Phi-3 Vision 128K — Kenyan Culture VQA\nAccuracy by Language")
ax.legend()
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            f"{acc:.1f}%", ha="center", va="bottom", fontsize=11)
plt.tight_layout()
plt.show()

# Sample images with English predictions
fig, axes = plt.subplots(1, 5, figsize=(15, 5))
eng_preds = all_results["English"]["predictions"]
for i, (sample, pred) in enumerate(zip(CUSTOM_DATA[:5], eng_preds[:5])):
    img = load_drive_image(sample["file_id"])
    axes[i].imshow(img)
    color  = "green" if pred["is_correct"] else "red"
    status = "✓" if pred["is_correct"] else "✗"
    axes[i].set_title(
        f"{status} Q{i+1}\nPred: {pred['predicted_label']} | Correct: {pred['correct_label']}",
        fontsize=8, color=color
    )
    axes[i].axis("off")
plt.suptitle(
    f"Phi-3 Vision 128K — Kenyan Culture Dataset (English) | Accuracy: {all_results['English']['accuracy']:.1f}% | Random baseline: 25%",
    fontsize=10
)
plt.tight_layout()
plt.show()
